# nb_03_ingest_to_index — Doc Intelligence → chunk → embed → AI Search

**Pipeline 2.** Works the `file_metadata` status queue: claims changed/new files, enforces ACLs,
extracts with Document Intelligence, chunks (with page numbers), embeds via Azure OpenAI, and
pushes to Azure AI Search — stamping `allowed_groups` for security trimming. Handles deletions,
re-ingest cleanup, retries/dead-letter, and logging.

See `PRODUCT_SPEC.md` sections 5, 7.3, 10, 11. (ACL-drift fast path + bounded parallelism are
added in nb_03 by Sprint 5.)


## Config + secrets


In [ ]:
import uuid, time, json, hashlib, io, os
from datetime import datetime, timezone
from pyspark.sql import functions as F
from delta.tables import DeltaTable

cfg = {r['key']: r['value'] for r in spark.table('config').collect()}
RUN_ID = str(uuid.uuid4())
ACL_BYPASS = cfg.get('acl_bypass_enabled', 'false').lower() == 'true'
SUPPORTED = set(cfg.get('supported_extensions', '').split(','))
MAX_RETRIES = int(cfg.get('max_retries', '3'))
CHUNK_SIZE = int(cfg.get('chunk_size', '1000'))
STRATEGY = cfg.get('chunk_strategy_version', 'v1')
EMB_DEPLOY = cfg['aoai_embedding_deployment']
MAX_CONC = int(cfg.get('max_concurrency', '8'))
BACKFILL = cfg.get('backfill_mode', 'false').lower() == 'true'
BATCH_LIMIT = int(cfg.get('backfill_batch_size', '500')) if BACKFILL else int(cfg.get('batch_size', '200'))

kv = f"https://{cfg['kv_name']}.vault.azure.net/"
DI_KEY = mssparkutils.credentials.getSecret(kv, cfg['kv_di_key_secret'])
AOAI_KEY = mssparkutils.credentials.getSecret(kv, cfg['kv_aoai_key_secret'])
SEARCH_KEY = mssparkutils.credentials.getSecret(kv, cfg['kv_search_admin_key_secret'])
print('run_id:', RUN_ID, '| acl_bypass:', ACL_BYPASS)


## ACL resolution (nearest-ancestor, efficient for deep paths)
Build a sorted prefix map once; each file resolves by walking up its path prefixes.


In [ ]:
acls_path = cfg.get('acls_file_path', 'Files/acls/acls.json')
try:
    raw = spark.read.text(acls_path, wholetext=True).collect()[0][0]
    ACL_MAP = {f['path'].rstrip('/'): f['groups'] for f in json.loads(raw).get('folders', [])}
except Exception as e:
    print('WARNING: no ACL file ->', e); ACL_MAP = {}

def resolve_groups(rel_path):
    """Return (groups, acl_version) using the nearest ancestor folder in ACL_MAP."""
    parts = rel_path.split('/')
    for i in range(len(parts) - 1, 0, -1):
        prefix = '/'.join(parts[:i])
        if prefix in ACL_MAP:
            groups = sorted(ACL_MAP[prefix])
            ver = hashlib.sha256(('|'.join(groups)).encode()).hexdigest()[:16]
            return groups, ver
    return [], None


## Path + client helpers


In [ ]:
def to_rel(file_path):
    """abfss://.../Files/... -> Files/... (and local mount path)."""
    marker = '/Files/'
    rel = 'Files/' + file_path.split(marker, 1)[1] if marker in file_path else file_path
    local = '/lakehouse/default/' + rel
    return rel, local

def di_client():
    from azure.core.credentials import AzureKeyCredential
    from azure.ai.documentintelligence import DocumentIntelligenceClient
    return DocumentIntelligenceClient(cfg['doc_intelligence_endpoint'], AzureKeyCredential(DI_KEY))

def aoai_client():
    from openai import AzureOpenAI
    return AzureOpenAI(azure_endpoint=cfg['aoai_endpoint'], api_key=AOAI_KEY,
                       api_version='2024-06-01')

def search_client():
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient
    return SearchClient(cfg['search_endpoint'], cfg['search_index_name'], AzureKeyCredential(SEARCH_KEY))


## Extract (Doc Intelligence) + chunk with page numbers


In [ ]:
def extract_pages(local_path):
    """Return list of (page_number, text). Raises on DI failure."""
    with open(local_path, 'rb') as f:
        data = f.read()
    poller = di_client().begin_analyze_document(cfg['doc_intelligence_model'], body=data,
                                                content_type='application/octet-stream')
    result = poller.result()
    pages = []
    for p in (result.pages or []):
        text = '\n'.join(l.content for l in (p.lines or []))
        pages.append((p.page_number, text))
    return pages

def chunk_pages(pages):
    """Chunk by page (no cross-chunk overlap), matching AI Search's default page chunking.
    Each page becomes one chunk tagged with its page number. A page longer than CHUNK_SIZE
    is split into multiple non-overlapping chunks that keep the same page number (safety cap
    so a huge page doesn't exceed embedding limits)."""
    out = []
    for page_number, text in pages:
        text = (text or '').strip()
        if not text:
            continue
        if len(text) <= CHUNK_SIZE:
            out.append((page_number, text))
        else:
            for i in range(0, len(text), CHUNK_SIZE):
                piece = text[i:i + CHUNK_SIZE].strip()
                if piece:
                    out.append((page_number, piece))
    return out


## Embed + build Search docs


In [ ]:
def embed(texts):
    resp = aoai_client().embeddings.create(model=EMB_DEPLOY, input=texts)
    return [d.embedding for d in resp.data]

def chunk_id(file_path, idx):
    h = hashlib.sha256(file_path.encode()).hexdigest()[:32]
    return f'{h}-{idx}'

def build_docs(file_path, file_name, ext, chunks, vectors, groups, indexed_utc):
    docs = []
    for idx, ((page, text), vec) in enumerate(zip(chunks, vectors)):
        docs.append({
            'chunk_id': chunk_id(file_path, idx),
            'file_path': file_path, 'file_name': file_name, 'file_extension': ext,
            'content': text, 'content_vector': vec,
            'page_number': int(page), 'chunk_index': idx,
            'allowed_groups': groups,
            'embedding_model': EMB_DEPLOY, 'chunk_strategy_version': STRATEGY,
            'indexed_utc': indexed_utc,
        })
    return docs


## Search write helpers (delete-by-file + batched upload)


In [ ]:
def delete_file_chunks(sc, file_path):
    """Delete all existing chunks for a file (deterministic ids) so re-ingest never duplicates."""
    ids = [d['chunk_id'] for d in sc.search(search_text='*', filter=f"file_path eq '{file_path}'",
                                            select=['chunk_id'], top=100000)]
    if ids:
        sc.delete_documents([{'chunk_id': i} for i in ids])
    return len(ids)

def upload_docs(sc, docs, batch=500):
    for i in range(0, len(docs), batch):
        sc.upload_documents(docs[i:i + batch])


## Status helpers


In [ ]:
def set_status(file_path, status, reason=None, inc_retry=False):
    now = datetime.now(timezone.utc)
    sets = {'process_status': F.lit(status), 'status_reason': F.lit(reason),
            'status_updated_utc': F.lit(now)}
    if inc_retry:
        sets['retry_count'] = F.coalesce(F.col('retry_count'), F.lit(0)) + 1
    (DeltaTable.forName(spark, 'file_metadata').update(
        condition=F.col('file_path') == F.lit(file_path), set=sets))

def log_success(file_path, chunks, pages, duration_ms):
    row = [(file_path, chunks, pages, duration_ms, EMB_DEPLOY, cfg['doc_intelligence_model'],
            RUN_ID, datetime.now(timezone.utc))]
    spark.createDataFrame(row, 'file_path string, chunks int, pages int, duration_ms long, embedding_model string, di_model string, run_id string, ts_utc timestamp')\
        .write.mode('append').saveAsTable('ingestion_log')

def log_skip(file_path, reason, detail):
    row = [(file_path, reason, str(detail)[:4000], RUN_ID, datetime.now(timezone.utc))]
    spark.createDataFrame(row, 'file_path string, reason string, detail string, run_id string, ts_utc timestamp')\
        .write.mode('append').saveAsTable('skipped_log')

def upsert_state(file_path, change_hash, acl_version, chunk_count):
    now = datetime.now(timezone.utc)
    src = spark.createDataFrame(
        [(file_path, change_hash, acl_version, chunk_count, EMB_DEPLOY, STRATEGY, now)],
        'file_path string, change_hash string, acl_version string, chunk_count int, embedding_model string, chunk_strategy_version string, indexed_utc timestamp')
    (DeltaTable.forName(spark, 'ingestion_state').alias('t')
       .merge(src.alias('s'), 't.file_path = s.file_path')
       .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


## Claim the work queue
The `pending`→`ingesting` transition is the claim; a run only processes what it claims.
Stale `ingesting` rows from a crashed/timed-out prior run (older than `ingesting_lease_minutes`)
are reclaimed so no work is stranded.


In [ ]:
WORK_STATUSES = ['new', 'changed', 'reingest', 'error']
LEASE_MIN = int(cfg.get('ingesting_lease_minutes', '120'))

# Recover stranded work: files stuck 'ingesting' past the lease -> back to the queue.
from datetime import timedelta
cutoff = datetime.now(timezone.utc) - timedelta(minutes=LEASE_MIN)
recovered = (DeltaTable.forName(spark, 'file_metadata').toDF()
             .where((F.col('process_status') == 'ingesting') &
                    (F.coalesce(F.col('status_updated_utc'), F.lit(datetime(1970,1,1))) < F.lit(cutoff)))
             .count())
if recovered:
    (DeltaTable.forName(spark, 'file_metadata').update(
        condition=(F.col('process_status') == 'ingesting') &
                  (F.coalesce(F.col('status_updated_utc'), F.lit(datetime(1970,1,1))) < F.lit(cutoff)),
        set={'process_status': F.lit('reingest'),
             'status_reason': F.lit('recovered_stale_ingesting'),
             'status_updated_utc': F.lit(datetime.now(timezone.utc))}))
    print(f'recovered {recovered} stranded ingesting file(s)')

# Handle deletions first: purge chunks, drop state, mark done.
deleted = [r['file_path'] for r in spark.table('file_metadata')
           .where(F.col('process_status') == 'deleted').select('file_path').collect()]
if deleted:
    sc = search_client()
    for fp in deleted:
        try:
            n = delete_file_chunks(sc, fp)
            spark.sql(f"DELETE FROM ingestion_state WHERE file_path = '{fp}'")
            print(f'purged {n} chunks for deleted file {fp}')
        except Exception as e:
            log_skip(fp, 'delete_error', e)

work = [r.asDict() for r in spark.table('file_metadata')
        .where(F.col('process_status').isin(WORK_STATUSES))
        .where(F.coalesce(F.col('retry_count'), F.lit(0)) < MAX_RETRIES)
        .select('file_path','file_name','file_extension','change_hash')
        .orderBy('file_path').limit(BATCH_LIMIT).collect()]
for w in work:
    set_status(w['file_path'], 'ingesting')
print(f'claimed for ingestion: {len(work)} (backfill={BACKFILL}, batch_limit={BATCH_LIMIT})')


## Process each file
Two phases: network-heavy work (DI → embed → Search) runs in a bounded thread pool
(`max_concurrency`); Delta status/state/log writes are applied serially on the driver to
avoid optimistic-concurrency conflicts on the tables.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def compute(w):
    """Pure network/compute; NO Delta writes. Returns a result dict."""
    fp = w['file_path']; ext = (w['file_extension'] or '').lower()
    rel, local = to_rel(fp)
    groups, acl_version = resolve_groups(rel)
    if not groups and not ACL_BYPASS:
        return {'fp': fp, 'status': 'skipped', 'reason': 'no_acl', 'detail': rel}
    if ext not in SUPPORTED:
        return {'fp': fp, 'status': 'skipped', 'reason': 'doc_intel_unsupported', 'detail': ext}
    t0 = time.time()
    try:
        pages = extract_pages(local)
        chunks = chunk_pages(pages)
        if not chunks:
            return {'fp': fp, 'status': 'skipped', 'reason': 'empty_extract', 'detail': rel}
        vectors = embed([c[1] for c in chunks])
        sc = search_client()
        delete_file_chunks(sc, fp)  # re-ingest cleanup
        now_iso = datetime.now(timezone.utc).isoformat()
        docs = build_docs(fp, w['file_name'], ext, chunks, vectors, groups, now_iso)
        upload_docs(sc, docs)
        return {'fp': fp, 'status': 'complete', 'change_hash': w['change_hash'],
                'acl_version': acl_version, 'chunks': len(docs), 'pages': len(pages),
                'duration_ms': int((time.time() - t0) * 1000)}
    except Exception as e:
        return {'fp': fp, 'status': 'error', 'error': str(e)}

def apply_result(r):
    """Serial Delta writes for one compute result."""
    fp = r['fp']
    if r['status'] == 'skipped':
        set_status(fp, 'skipped', r['reason']); log_skip(fp, r['reason'], r['detail']); return 'skipped'
    if r['status'] == 'complete':
        upsert_state(fp, r['change_hash'], r['acl_version'], r['chunks'])
        set_status(fp, 'complete')
        log_success(fp, r['chunks'], r['pages'], r['duration_ms']); return 'complete'
    # error path w/ retry -> dead_letter
    rc = spark.table('file_metadata').where(F.col('file_path') == fp)\
        .select('retry_count').collect()[0][0] or 0
    if rc + 1 >= MAX_RETRIES:
        set_status(fp, 'dead_letter', r['error'][:500], inc_retry=True); log_skip(fp, 'dead_letter', r['error'])
    else:
        set_status(fp, 'error', r['error'][:500], inc_retry=True); log_skip(fp, 'doc_intel_error', r['error'])
    return 'error'

results = {}
with ThreadPoolExecutor(max_workers=MAX_CONC) as pool:
    for r in pool.map(compute, work):
        outcome = apply_result(r)
        results[outcome] = results.get(outcome, 0) + 1
print('run complete:', results)


## Summary


In [ ]:
spark.table('file_metadata').groupBy('process_status').count().orderBy('process_status').show()
print('--- skipped this run ---')
spark.table('skipped_log').where(F.col('run_id') == RUN_ID).show(truncate=False)
